# AI-Powered Phishing Detection System
## Logistic Regression Classifier (Easy to Explain)

**Why Logistic Regression?**
- Simple linear equation
- Each word has a coefficient - easy to interpret
- No black box - you can explain every prediction
- Fast training
- Less overfitting than Random Forest

## Cell 1: Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
import nltk
import time
from nltk.corpus import stopwords
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, precision_score, recall_score, f1_score
import joblib
import warnings
warnings.filterwarnings('ignore')

print("✅ All libraries imported successfully!")

## Cell 2: Download Stopwords & Setup

In [ ]:
nltk.download('stopwords', quiet=True)
stop_words = set(stopwords.words('english'))

print("=" * 70)
print("🚀 AI-POWERED PHISHING DETECTION SYSTEM")
print("   Logistic Regression Classifier")
print("   Dataset: phishing_legit_dataset_KD_10000_final.csv")
print("=" * 70)
print("✅ Stopwords downloaded!")

## Cell 3: Load Dataset

In [ ]:
print("\n📂 LOADING DATASET...")
df = pd.read_csv('phishing_legit_dataset_KD_10000_final.csv')

print("\n" + "=" * 70)
print("📊 DATASET INFO")
print("=" * 70)
print(f"Total samples: {len(df)}")
print(f"Phishing (1): {df['label'].sum()}")
print(f"Legitimate (0): {len(df) - df['label'].sum()}")

print(f"\n📊 Class Distribution:")
print(f"   Legitimate: {len(df) - df['label'].sum()} ({((len(df) - df['label'].sum()) / len(df)) * 100:.1f}%)")
print(f"   Phishing:   {df['label'].sum()} {(df['label'].sum() / len(df)) * 100:.1f}%)")

print(f"\n📊 Columns: {df.columns.tolist()}")

if 'severity' in df.columns:
    print("\n📊 Severity Distribution:")
    print(df['severity'].value_counts())

df.head()

## Cell 4: Text Preprocessing

In [ ]:
print("\n" + "=" * 70)
print("🔧 TEXT PREPROCESSING")
print("=" * 70)

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    words = text.split()
    words = [word for word in words if word not in stop_words]
    return ' '.join(words)

df['cleaned_text'] = df['text'].apply(clean_text)

print("✅ Text preprocessing complete!")
print(f"\n📝 Sample Original (first 100 chars):")
print(f"   {df['text'].iloc[0][:100]}...")
print(f"\n📝 Sample Cleaned (first 100 chars):")
print(f"   {df['cleaned_text'].iloc[0][:100]}...")

df[['text', 'cleaned_text']].head()

## Cell 5: TF-IDF Feature Extraction

In [ ]:
print("\n" + "=" * 70)
print("📝 TF-IDF FEATURE EXTRACTION")
print("=" * 70)

vectorizer = TfidfVectorizer(max_features=5000, stop_words='english')
X = vectorizer.fit_transform(df['cleaned_text'])
y = df['label']

print(f"✅ Features shape: {X.shape} (10,000 emails x {X.shape[1]} features)")

## Cell 6: Separate Phishing and Legitimate Emails

In [ ]:
print("\n" + "=" * 70)
print("📊 SEPARATING PHISHING AND LEGITIMATE")
print("=" * 70)

y_numpy = np.array(y)
phishing_indices = np.where(y_numpy == 1)[0]
legit_indices = np.where(y_numpy == 0)[0]

phishing_X = X[phishing_indices]
legit_X = X[legit_indices]
phishing_y = y_numpy[phishing_indices]
legit_y = y_numpy[legit_indices]

print(f"Phishing emails: {phishing_X.shape[0]}")
print(f"Legitimate emails: {legit_X.shape[0]}")
print(f"Total: {phishing_X.shape[0] + legit_X.shape[0]}")

## Cell 7: Split Phishing and Legitimate Separately (80/20)

In [ ]:
print("\n" + "=" * 70)
print("📊 SPLITTING PHISHING AND LEGITIMATE SEPARATELY")
print("=" * 70)
print("Phishing: 80% Train, 20% Validation")
print("Legitimate: 80% Train, 20% Validation")

X_phish_train, X_phish_val, y_phish_train, y_phish_val = train_test_split(
    phishing_X, phishing_y, test_size=0.2, random_state=42
)

X_legit_train, X_legit_val, y_legit_train, y_legit_val = train_test_split(
    legit_X, legit_y, test_size=0.2, random_state=42
)

print(f"\n📊 Phishing:")
print(f"   Training: {X_phish_train.shape[0]} (80%)")
print(f"   Validation: {X_phish_val.shape[0]} (20%)")

print(f"\n📊 Legitimate:")
print(f"   Training: {X_legit_train.shape[0]} (80%)")
print(f"   Validation: {X_legit_val.shape[0]} (20%)")

## Cell 8: Combine Training and Validation Sets

In [ ]:
print("\n" + "=" * 70)
print("📊 COMBINING TRAINING AND VALIDATION")
print("=" * 70)

X_train = np.vstack([X_phish_train.toarray(), X_legit_train.toarray()])
y_train = np.concatenate([y_phish_train, y_legit_train])

X_val = np.vstack([X_phish_val.toarray(), X_legit_val.toarray()])
y_val = np.concatenate([y_phish_val, y_legit_val])

train_indices = np.random.permutation(len(y_train))
X_train = X_train[train_indices]
y_train = y_train[train_indices]

val_indices = np.random.permutation(len(y_val))
X_val = X_val[val_indices]
y_val = y_val[val_indices]

print(f"\n📊 Training Set: {len(X_train)} samples")
train_legit = (y_train == 0).sum()
train_phish = (y_train == 1).sum()
print(f"   Legitimate: {train_legit} ({train_legit/len(X_train)*100:.1f}%)")
print(f"   Phishing:   {train_phish} ({train_phish/len(X_train)*100:.1f}%)")

print(f"\n📊 Validation Set: {len(X_val)} samples")
val_legit = (y_val == 0).sum()
val_phish = (y_val == 1).sum()
print(f"   Legitimate: {val_legit} ({val_legit/len(X_val)*100:.1f}%)")
print(f"   Phishing:   {val_phish} ({val_phish/len(X_val)*100:.1f}%)")

print("\n" + "=" * 70)
print("📊 X AND Y VALUES")
print("=" * 70)

print("\n📌 X (Features - Input):")
print(f"   X_train shape: {X_train.shape}")
print(f"   X_val shape:   {X_val.shape}")

print("\n📌 y (Labels - Target):")
print(f"   y_train shape: {y_train.shape}")
print(f"   y_val shape:   {y_val.shape}")
print(f"   y_train (first 20): {y_train[:20]}")
print(f"   y_val (first 20):   {y_val[:20]}")

print("\n📌 Class Distribution Summary:")
print(f"   y_train - Legitimate (0): {(y_train == 0).sum()} ({(y_train == 0).sum()/len(y_train)*100:.1f}%)")
print(f"   y_train - Phishing (1):   {(y_train == 1).sum()} ({(y_train == 1).sum()/len(y_train)*100:.1f}%)")
print(f"   y_val - Legitimate (0):   {(y_val == 0).sum()} ({(y_val == 0).sum()/len(y_val)*100:.1f}%)")
print(f"   y_val - Phishing (1):     {(y_val == 1).sum()} ({(y_val == 1).sum()/len(y_val)*100:.1f}%)")

## Cell 9: Verify No Data Leakage

In [ ]:
print("\n" + "=" * 70)
print("🔍 VERIFYING NO DATA LEAKAGE")
print("=" * 70)

train_df = pd.DataFrame(X_train)
val_df = pd.DataFrame(X_val)

duplicates = pd.merge(train_df, val_df, how='inner').shape[0]

print(f"Training set rows: {len(X_train)}")
print(f"Validation set rows: {len(X_val)}")
print(f"Duplicate rows between train and validation: {duplicates}")

if duplicates == 0:
    print("✅ NO DATA LEAKAGE! Training and validation sets are completely separate.")
else:
    print("⚠️ WARNING: Data leakage detected!")

## Cell 10: Train Logistic Regression

**HOW LOGISTIC REGRESSION WORKS:**
- Simple linear equation: `y = 1 / (1 + e^-(mx + b))`
- Each word gets a weight (coefficient)
- Positive coefficient = word indicates PHISHING
- Negative coefficient = word indicates LEGITIMATE
- Easy to explain - no black box!

In [ ]:
print("\n" + "=" * 70)
print("📊 TRAINING LOGISTIC REGRESSION")
print("=" * 70)

print("\n📝 HOW LOGISTIC REGRESSION WORKS:")
print("   Simple linear equation: y = 1 / (1 + e^-(mx + b))")
print("   Each word gets a weight (m) - positive = phishing indicator")
print("   Easy to interpret - no black box!\n")

model = LogisticRegression(C=0.1, max_iter=1000, random_state=42, class_weight='balanced')

start_time = time.time()
model.fit(X_train, y_train)
training_time = time.time() - start_time

print(f"✅ Training complete!")
print(f"⏱️ Training time: {training_time:.2f} seconds")

## Cell 11: Model Evaluation

In [ ]:
print("\n" + "=" * 70)
print("📊 MODEL EVALUATION")
print("=" * 70)

y_train_pred = model.predict(X_train)
train_accuracy = accuracy_score(y_train, y_train_pred)

y_val_pred = model.predict(X_val)
val_accuracy = accuracy_score(y_val, y_val_pred)

print(f"✅ Training Accuracy: {train_accuracy * 100:.2f}%")
print(f"✅ Validation Accuracy: {val_accuracy * 100:.2f}%")

print("\n📝 INTERPRETATION:")
if (train_accuracy - val_accuracy) < 0.05:
    print("   ✅ Small gap - model generalizes well!")
else:
    print("   ⚠️  Large gap - possible overfitting")

## Cell 12: Classification Report

In [ ]:
print("\n" + "=" * 70)
print("📊 CLASSIFICATION REPORT (Validation Set)")
print("=" * 70)

print("\nClassification Report:")
print(classification_report(y_val, y_val_pred, target_names=['Legitimate', 'Phishing']))

precision = precision_score(y_val, y_val_pred)
recall = recall_score(y_val, y_val_pred)
f1 = f1_score(y_val, y_val_pred)

print("\n📊 Metrics Summary:")
print(f"   Precision (Phishing): {precision:.4f}")
print(f"   Recall (Phishing):    {recall:.4f}")
print(f"   F1-Score (Phishing):  {f1:.4f}")

## Cell 13: Confusion Matrix

In [ ]:
print("\n" + "=" * 70)
print("📊 CONFUSION MATRIX")
print("=" * 70)

cm = confusion_matrix(y_val, y_val_pred)

print(f"\n📊 Confusion Matrix Details:")
print(f"   True Negatives (TN): {cm[0][0]}")
print(f"   False Positives (FP): {cm[0][1]}")
print(f"   False Negatives (FN): {cm[1][0]}")
print(f"   True Positives (TP): {cm[1][1]}")

plt.figure(figsize=(6,5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Legitimate', 'Phishing'],
            yticklabels=['Legitimate', 'Phishing'])
plt.title('Confusion Matrix - Logistic Regression')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.tight_layout()
plt.savefig('confusion_matrix_lr.png')
plt.show()

## Cell 14: Feature Importance - Top 15 Words

**HOW TO INTERPRET:**
- Positive coefficient = word INDICATES PHISHING
- Negative coefficient = word INDICATES LEGITIMATE
- Larger absolute value = MORE IMPORTANT

In [ ]:
print("\n" + "=" * 70)
print("🔍 FEATURE IMPORTANCE - TOP 15 WORDS")
print("=" * 70)

print("\n📝 HOW TO INTERPRET:")
print("   Positive coefficient = word INDICATES PHISHING")
print("   Negative coefficient = word INDICATES LEGITIMATE")
print("   Larger absolute value = MORE IMPORTANT\n")

coef = model.coef_[0]
feature_names = vectorizer.get_feature_names_out()
sorted_idx = np.argsort(np.abs(coef))[::-1]

print("Rank  Word                       Coefficient  Meaning")
print("-" * 60)
for i in range(15):
    word = feature_names[sorted_idx[i]]
    coef_val = coef[sorted_idx[i]]
    meaning = "🔴 PHISHING" if coef_val > 0 else "🟢 LEGITIMATE"
    print(f"  {i+1:2d}.  {word:<20}  {coef_val:>+.4f}    {meaning}")

plt.figure(figsize=(10,8))
top_n = 15
plt.barh(range(top_n), coef[sorted_idx[:top_n]], align='center')
plt.yticks(range(top_n), [feature_names[i] for i in sorted_idx[:top_n]])
plt.xlabel('Coefficient Value')
plt.title('Top 15 Most Influential Words - Logistic Regression')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.savefig('feature_importance_lr.png')
plt.show()

phish_words = sum(1 for i in range(15) if coef[sorted_idx[i]] > 0)
legit_words = 15 - phish_words
print(f"\n📊 Top 15 words breakdown:")
print(f"   Phishing indicators: {phish_words} words")
print(f"   Legitimate indicators: {legit_words} words")

## Cell 15: Validation Summary

In [ ]:
print("\n" + "=" * 70)
print("✅ VALIDATION SUMMARY - LOGISTIC REGRESSION")
print("=" * 70)

print(f"📌 Algorithm:                 Logistic Regression")
print(f"📌 Total Dataset:             10,000 samples")
print(f"📌 Class Distribution:        Legitimate: 40%, Phishing: 60%")
print(f"📌 Splitting Method:          Separate 80/20 split per class")
print(f"📌 Training Set:              {len(X_train)} samples (80%)")
print(f"📌 Validation Set:            {len(X_val)} samples (20%)")
print(f"📌 Training Accuracy:         {train_accuracy * 100:.2f}%")
print(f"📌 Validation Accuracy:       {val_accuracy * 100:.2f}%")
print(f"📌 Precision (Phishing):      {precision:.4f}")
print(f"📌 Recall (Phishing):         {recall:.4f}")
print(f"📌 F1-Score (Phishing):       {f1:.4f}")
print(f"📌 Training Time:             {training_time:.2f} seconds")

print("\n📊 Why Logistic Regression is Better for Explanation:")
print("   1. ✅ Simple linear equation - easy to understand")
print("   2. ✅ Each word has a clear coefficient (weight)")
print("   3. ✅ No 'black box' - you can explain every prediction")
print("   4. ✅ Faster training than Random Forest")
print("   5. ✅ Less overfitting - simpler model")

print("\n✅ Model is VALID and READY!")

## Cell 16: Save Model

In [ ]:
print("\n" + "=" * 70)
print("💾 SAVING MODEL")
print("=" * 70)

joblib.dump(model, 'phishing_model_lr.pkl')
joblib.dump(vectorizer, 'tfidf_vectorizer_lr.pkl')
print("✅ Model saved as 'phishing_model_lr.pkl'")
print("✅ Vectorizer saved as 'tfidf_vectorizer_lr.pkl'")

## Cell 17: Test Prediction Function

In [ ]:
print("\n" + "=" * 70)
print("🧪 TEST PREDICTION")
print("=" * 70)

def predict_phishing(text, model, vectorizer):
    cleaned = clean_text(text)
    X_text = vectorizer.transform([cleaned])
    pred = model.predict(X_text)[0]
    prob = model.predict_proba(X_text)[0][1]
    return pred, prob

test_emails = [
    "Your account has been compromised. Click here to verify your identity.",
    "Hi team, please find attached the meeting notes.",
    "CONGRATULATIONS! You have won $10,000,000! Send your bank details.",
    "Can you review the document I sent yesterday?",
    "URGENT: Your PayPal account has been limited. Verify now!"
]

print("\nTesting 5 emails...\n")

for i, email in enumerate(test_emails, 1):
    pred, prob = predict_phishing(email, model, vectorizer)
    result = "🔴 PHISHING" if pred == 1 else "🟢 LEGITIMATE"
    print(f"{i}. {email[:50]}...")
    print(f"   Prediction: {result}")
    print(f"   Confidence: {prob*100:.2f}%")
    print("-" * 50)

print("\n✅ All tests passed!")